<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 6 — Fine-tuning de BERT</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">Un backbone, tres cabezas — con datasets reales y demo sobre su corpus</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.


> ⚙️ **Entorno: Google Colab con GPU T4 (free tier).** Entorno de ejecución → Cambiar tipo → **GPU**. Los tres modelos (BETO 110M, Sentence-BERT) caben de sobra en los 15 GB de la T4 usando `fp16`. Esta versión entrena con **datasets reales de HuggingFace** (no con el corpus de juguete): `tweet_sentiment_multilingual` para clasificación y `conll2002` para NER. El corpus chiapaneco se usa solo para la **inferencia final** de cada parte. Las celdas de *liberar memoria* permiten correr A → B → C en una sola sesión sin saturar la VRAM.


## Objetivo

Tres partes, el mismo BERT preentrenado en español como base. **A)** Afinar un encoder de oraciones (Sentence-BERT) con sus pares del Lab 3. **B)** Clasificar **sentimiento** con un dataset real en español. **C)** **NER** con CoNLL-2002. Cada parte cierra **usando el modelo afinado** sobre el corpus chiapaneco, y libera memoria antes de la siguiente.


## 0 · Setup, GPU y utilidades

In [ ]:
# Instalación (Colab). Reiniciar el entorno si lo pide tras instalar.
!pip install -q transformers sentence-transformers datasets seqeval accelerate

import gc, math, json
import numpy as np
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — activen el runtime de GPU')

def liberar_memoria():
    """Libera RAM/VRAM. Llamar tras borrar (del) las variables del entrenamiento previo."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f'VRAM en uso: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# El corpus chiapaneco (del Lab 1) se usa SOLO para la inferencia final de cada parte.
with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)
with open('corpus_crudo.json', encoding='utf-8') as fh:
    crudo = {d['id']: d['texto'] for d in json.load(fh)}
ids = [d['id'] for d in corpus]; titulos = {d['id']: d['titulo'] for d in corpus}
print(len(corpus), 'documentos del corpus cargados (para inferencia).')

---
## Parte A · Embeddings con Sentence-BERT (datos: sus qrels del Lab 3)

> **Por qué aquí sí usamos el corpus.** El fine-tuning de embeddings necesita pares *de dominio* (consulta ↔ documento relevante). Esos pares salen de **sus** qrels del Lab 3: es el cierre del arco de la unidad, donde su juicio de relevancia se vuelve señal de entrenamiento. **Advertencia metodológica:** con ~5 consultas esto es una *demostración del método*, no un experimento — la mejora puede ser chica o ruidosa. Lo evaluable es el pipeline correcto, no el número.


**A.1** Línea base: carguen un Sentence-BERT en español y midan su buscador **sin afinar** con su nDCG del Lab 3.

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import numpy as np
modelo = SentenceTransformer('hiiamsid/sentence_similarity_spanish_es')
def emb_corpus(corpus):
    return {doc['id']: modelo.encode(doc['texto']) for doc in corpus}

def buscar(consulta, modelo, embeddings, k=5):
    vec_q = modelo.encode(consulta)
    sims = [(doc_id, np.dot(vec_q, vec) / (np.linalg.norm(vec_q)*np.linalg.norm(vec))) for doc_id, vec in embeddings.items()]
    sims.sort(key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in sims[:k]]

print('Funciones de búsqueda y vectorización definidas. El nDCG requiere los qrels del lab 3.')


**A.2** Pares (consulta, documento relevante) desde qrels + fine-tuning con `MultipleNegativesRankingLoss`.

In [ ]:
# train_examples = [InputExample(texts=[q, d]) for q, d in pares_positivos] # Requiere qrels
# train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
# train_loss = losses.MultipleNegativesRankingLoss(model=modelo)
# modelo.fit(train_objectives=[(train_dataloader, train_loss)], epochs=2, warmup_steps=100)
print('Entrenamiento de SBERT configurado en código comentado (requiere pares de consulta-doc positivos).')


_Reporten los tres nDCG (FastText, SBERT base, SBERT afinado) y comenten:_ 

**A.3 · Uso del modelo afinado.** Busquen con una consulta nueva usando el encoder ya entrenado.

In [ ]:
# embeddings_afinados = emb_corpus(corpus)
# res = buscar('cultivo de café sostenible en la región', modelo, embeddings_afinados)
# print('Resultados SBERT afinado:', res)
print('Ejemplo de búsqueda preparado.')


**Liberar memoria** antes del siguiente entrenamiento (clave en T4).

In [ ]:
# Borren las variables del modelo de esta parte y liberen VRAM:
del modelo
liberar_memoria()

---
## Parte B · Clasificación de sentimiento (dataset real en español)

Entrenamos con **`cardiffnlp/tweet_sentiment_multilingual`** (config `spanish`): miles de ejemplos etiquetados (negativo / neutral / positivo) con splits oficiales train/validation/test. *Si el id cambiara en HF, es la única línea a ajustar; alternativas: `muchocine` (reseñas de cine, 5 clases).*


**B.1** Cargar el dataset y (en T4) submuestrear train para que entrene en minutos.

In [ ]:
from datasets import load_dataset
ds = load_dataset('cardiffnlp/tweet_sentiment_multilingual', 'spanish')
labels = ds['train'].features['label'].names
id2lab = {i: label for i, label in enumerate(labels)}
lab2id = {label: i for i, label in enumerate(labels)}
ds_tr = ds['train'].shuffle(seed=42).select(range(2000))
ds_te = ds['test']
print('Clases:', labels)


**B.2** Tokenizar con el tokenizer de BETO.

In [ ]:
from transformers import AutoTokenizer
CKPT = 'dccuchile/bert-base-spanish-wwm-cased'
tok = AutoTokenizer.from_pretrained(CKPT)
def tokenize_fn(examples):
    return tok(examples['text'], truncation=True, padding='max_length', max_length=128)
ds_tr = ds_tr.map(tokenize_fn, batched=True)
ds_te = ds_te.map(tokenize_fn, batched=True)
print('Datasets tokenizados.')


**B.3** Fine-tuning con `Trainer` (LR 2e-5, pocas épocas, `fp16` para la T4).

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# Descomentar para entrenar en Colab con GPU:
# model = AutoModelForSequenceClassification.from_pretrained(CKPT, num_labels=len(labels), id2label=id2lab, label2id=lab2id)
# def compute_metrics(eval_pred):
#     preds, labels = eval_pred
#     preds = np.argmax(preds, axis=1)
#     return {'accuracy': accuracy_score(labels, preds), 'f1_macro': f1_score(labels, preds, average='macro')}
# args = TrainingArguments('test_trainer', learning_rate=2e-5, fp16=True, num_train_epochs=3)
# trainer = Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_te, compute_metrics=compute_metrics)
# trainer.train()
# trainer.evaluate()
print('Trainer de clasificación configurado.')


**B.4** Análisis de errores: matriz de confusión y reporte por clase.

In [ ]:
# from sklearn.metrics import classification_report, confusion_matrix
# preds = trainer.predict(ds_te)
# preds_labels = np.argmax(preds.predictions, axis=1)
# print(classification_report(ds_te['label'], preds_labels, target_names=labels))
# print(confusion_matrix(ds_te['label'], preds_labels))
print('Evaluación de clasificación lista para ejecutarse tras el entrenamiento.')


_¿Qué clase es la más difícil? ¿Accuracy o F1-macro es mejor criterio aquí? ¿Por qué?_ 

**B.5 · Uso del modelo afinado** — transferencia al corpus chiapaneco. El modelo se entrenó con tuits; veamos qué predice sobre noticias.

In [ ]:
from transformers import pipeline
# clf = pipeline('text-classification', model=model, tokenizer=tok)
# frases = ['El clima en Chiapas es excelente', 'Hubo una grave sequía', 'La situación es neutral']
# print(clf(frases))
# for d in corpus[:3]:
#     print(d['id'], clf(d['texto'][:512]))
print('Pipeline de clasificación configurado.')


**Liberar memoria** antes de la Parte C.

In [ ]:
del model            # y el trainer/pipeline si los nombraron distinto
liberar_memoria()

---
## Parte C · NER con CoNLL-2002 (español)

Entrenamos NER con **`conll2002`** config `es`, el estándar en español: esquema BIO con PER/ORG/LOC/MISC y miles de oraciones anotadas. *Si la carga falla por la versión de `datasets`, prueben `load_dataset('conll2002','es', trust_remote_code=True)` o el espejo `eriktks/conll2002`.*


**C.1** Cargar el dataset y leer el esquema de etiquetas desde sus features.

In [ ]:
from datasets import load_dataset
conll = load_dataset('conll2002', 'es', trust_remote_code=True)
labels_ner = conll['train'].features['ner_tags'].feature.names
id2lab_ner = {i: lab for i, lab in enumerate(labels_ner)}
lab2id_ner = {lab: i for i, lab in enumerate(labels_ner)}
print('Etiquetas NER:', labels_ner)


**C.2 — el corazón del lab.** Tokenizar y **alinear etiquetas con subpalabras**: la etiqueta va a la **primera** subpalabra de cada palabra; las demás (y `[CLS]`/`[SEP]`) se marcan con `-100` para ignorarlas en la pérdida.

In [ ]:
from transformers import AutoTokenizer
CKPT = 'dccuchile/bert-base-spanish-wwm-cased'
tok = AutoTokenizer.from_pretrained(CKPT)
def tokeniza_y_alinea(batch):
    tokenized_inputs = tok(batch['tokens'], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(batch['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

conll_tok = conll.map(tokeniza_y_alinea, batched=True, remove_columns=conll['train'].column_names)
print('Dataset CoNLL tokenizado y alineado.')


**C.3** Fine-tuning con `AutoModelForTokenClassification` (`fp16`, T4).

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
# Descomentar para entrenar NER en Colab:
# model_ner = AutoModelForTokenClassification.from_pretrained(CKPT, num_labels=len(labels_ner), id2label=id2lab_ner, label2id=lab2id_ner)
# collator = DataCollatorForTokenClassification(tokenizer=tok)
# args_ner = TrainingArguments('ner_trainer', fp16=True, num_train_epochs=2)
# trainer_ner = Trainer(model=model_ner, args=args_ner, train_dataset=conll_tok['train'].select(range(1000)), eval_dataset=conll_tok['validation'].select(range(200)), data_collator=collator)
# trainer_ner.train()
print('Trainer de NER configurado.')


**C.4** Evaluación con **seqeval** (a nivel de entidad, no de token).

In [ ]:
from seqeval.metrics import classification_report as seq_report
import numpy as np
# preds, labels, _ = trainer_ner.predict(conll_tok['validation'].select(range(200)))
# preds_arg = np.argmax(preds, axis=2)
# true_preds = [[labels_ner[p] for (p, l) in zip(pred, label) if l != -100] for pred, label in zip(preds_arg, labels)]
# true_labels = [[labels_ner[l] for (p, l) in zip(pred, label) if l != -100] for pred, label in zip(preds_arg, labels)]
# print(seq_report(true_labels, true_preds))
# Explicación: No se usa accuracy porque la inmensa mayoría de los tokens son de clase 'O', lo cual infla artificialmente el accuracy y no refleja si el modelo detecta bien las entidades completas.
print('Evaluación con seqeval preparada.')


_¿Por qué seqeval y no accuracy por token? ¿Qué tipo de entidad cuesta más?_ 

**C.5 · Uso del modelo afinado** — cierre del círculo: extraer entidades del **corpus chiapaneco**.

In [ ]:
from transformers import pipeline
# ner_pipe = pipeline('ner', model=model_ner, tokenizer=tok, aggregation_strategy='simple')
# for d in corpus[:3]:
#     print(d['id'], ner_pipe(d['texto'][:512]))
print('Pipeline NER configurado para extraer sobre el corpus.')


**Liberar memoria** al terminar.

In [ ]:
del model
liberar_memoria()

---
## Síntesis

Las tres partes usaron **el mismo BERT preentrenado** y solo cambiaron la cabeza y los datos: cabeza siamesa con sus qrels (A), `[CLS]` + lineal con un dataset de sentimiento real (B), y una etiqueta por token con CoNLL-2002 (C). Cada modelo afinado se aplicó **sobre su propio corpus**, y entre entrenamientos se liberó memoria para sostener la sesión en una T4. Ese paradigma —preentrenar una vez, adaptar barato— es la base de los sistemas RAG de la Unidad 3.


## Entregables — Lab 6
- [ ] **A:** fine-tuning de Sentence-BERT + los tres nDCG + búsqueda con el modelo afinado.
- [ ] **B:** clasificación con dataset real + matriz de confusión + inferencia sobre el corpus.
- [ ] **C:** NER con CoNLL-2002 + alineación de subpalabras + seqeval + extracción sobre el corpus.
- [ ] Celdas de liberación de memoria ejecutadas entre partes.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
